# CS781 B1: hierarchy and controlled head comparisons

One independently trained model per prediction. Own Phase1 B1 weights and tokenizer only. This segment compares GRU-only and transformer-only document heads on the same 512-token/16-window representation, initialized independently from own Phase1 B1 weights. These are B1 head ablations, not different course architecture families. Encoder growth and optional FFN expansion are implemented but are not launched before earlier evidence.

Three-hour training segments save exact resumable state. A budget boundary is **paused**, not converged. Published outputs must be downloaded and verified before another segment continues. No live `/kaggle/working` checkpoint is claimed as a D-drive backup. All experiments are after the original course deadline. Accuracy is measured, never guaranteed.


In [ ]:
import os,sys,time,subprocess,importlib.util
from pathlib import Path
SESSION_STARTED=time.time()
SESSION_END=SESSION_STARTED+4.2*3600
os.environ.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
needed={'ijson':'ijson>=3.2,<4','sklearn':'scikit-learn>=1.4,<2','tokenizers':'tokenizers'}
missing=[p for m,p in needed.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('B1_STUDY_BOOT',{'session_start':SESSION_STARTED,'session_end':SESSION_END})


In [ ]:
import base64,hashlib,io,json,zipfile,shutil
EXPECTED_VENDOR={'phase3_rr/__init__.py': 'f38a2af63c8fe34b0e5fe200c4a9c71fb161c9ad6a37bf46ec7307eb24d00ae6', 'phase3_rr/archive_inputs.py': '900109016e17f4170100f04a630b7259e41eee51442d04d015b40a85d0512391', 'phase3_rr/common.py': 'f808cde35564328d8f377dabb79da21e8ef1001623d9897d7b789fd2251e6535', 'phase3_rr/data.py': '00084b779497b1480c9574583faef1b6700a7bd92c4a1ff2fe5c1d1cd0ffc02a', 'phase3_rr/models.py': '14fc38d447e1fcb8d663d00932b24344c74d7c383cf7b543a8b90cc4e97c4eea', 'phase3_rr/train.py': 'db1532039e21f531b81f15b568a7ab97ce86779df5df4b5b0ff1f4299fbd4f63', 'phase3_rr/vendor/__init__.py': '4e813963bf49a55dffeba982788a12a8d854c7e7a4e6ed19b300b4ea35cef565', 'phase3_rr/vendor/backbones/cs781/__init__.py': '4a3613ae45d376f074874f3924962127b635a12fc6a03c5cf2c2d2e9cc3c1c49', 'phase3_rr/vendor/backbones/cs781/checkpoints.py': 'f70275cc12a1ca3bda33c4103dc2f697b0a374f8dcd5a3f30ab9f9f3f3b08418', 'phase3_rr/vendor/backbones/cs781/cli.py': 'e6695a4ec962094bf8fba96d61e61182f421440af61475644ae5dfe9223bfd8f', 'phase3_rr/vendor/backbones/cs781/cli_b.py': '471b80b2b4485a7bbc13a196fbfa452c7faa536fce85c7f9293742012248aa33', 'phase3_rr/vendor/backbones/cs781/cli_c.py': '0e6b8d4aa1b942aeb214881d1d9074e524ce90dfed641b5438a41601874c3aa6', 'phase3_rr/vendor/backbones/cs781/contracts.py': 'b4d17bf29617b79bff3c3507cf01245dbe16ad550c2c394a15aaddedd0733689', 'phase3_rr/vendor/backbones/cs781/corpus.py': '5706941f90284ba4481524a6ffe93c9f108bfcadcb0845107a107c57891098bf', 'phase3_rr/vendor/backbones/cs781/evaluate.py': '9ca3eb30f79592adfce063cdb47a34cb21a84b33b7406c8e888ce980267dba2a', 'phase3_rr/vendor/backbones/cs781/examples.py': '372e9d40984f66bda6873dbd87382416544eee410984651c27a7d000c38b1eba', 'phase3_rr/vendor/backbones/cs781/examples_alt.py': '7260c5b156dcd046c23545ab851c25014aef76a81ca73b612216de80a4c83052', 'phase3_rr/vendor/backbones/cs781/generate.py': '9a3bc31bd2a5e0e7fdb342e81e0fc20c031beab13711dc3bbf96b5728dcc8d03', 'phase3_rr/vendor/backbones/cs781/generate_b.py': 'e92ed53fcf2f6853eda5cc8a48bdacac0c2b92478e4677d4161dc3b4f6eba6ba', 'phase3_rr/vendor/backbones/cs781/generate_c.py': '43be46a4714d76a835f83a792b72210ceb80b889ce30675e13555243bc7d598e', 'phase3_rr/vendor/backbones/cs781/io.py': '6f7c8510cff6eee2d6cc82dbb77cd0d3d3cd7063670c0da37363ca7d600f45e4', 'phase3_rr/vendor/backbones/cs781/models/__init__.py': '0da8971507f5266ea3b85759054329394979d08d7f8e4896472fd471f78d3d49', 'phase3_rr/vendor/backbones/cs781/models/a.py': 'b5eb47496093388acf0798fa44fe2155116bd30a3fd1a9fbec3f6a71b5dca268', 'phase3_rr/vendor/backbones/cs781/models/b.py': '9919cc6a3ac87fb080acf6ac2be2df84bfb0d1d6d314e53fcf6c977bde14c057', 'phase3_rr/vendor/backbones/cs781/models/c.py': 'd9c16e1b53c23db7fd2a2be3e83f09c30642012fd31a63dc8282b7e39dc98147', 'phase3_rr/vendor/backbones/cs781/own_bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/cs781/prediction_progress.py': '3d35956bdecc1f88890010be3cd15527bd33ce463ff48b2a304db470f0a505c4', 'phase3_rr/vendor/backbones/cs781/prepare.py': 'efb7368de000b0e9f44595072b0f7f146e1809ef6f34e799859b14b33de79f9e', 'phase3_rr/vendor/backbones/cs781/splits.py': '2434ecd26e762919b66d4586ab530615c578d677a385bb1c78eb06f4d2cfee83', 'phase3_rr/vendor/backbones/cs781/submission.py': 'c36de151bccce0f844827c33087ec9c1b3163cb2ed7a32e45d7bf198cad1063d', 'phase3_rr/vendor/backbones/cs781/tokenizer.py': 'e838c764e9dc2848c1ea04f9f51b261aab1afe5607cfe8cf7322ab5984555558', 'phase3_rr/vendor/backbones/cs781/train.py': 'a2367df0b50efb36660b067940356b953340240483feb787d2108ee4258d6588', 'phase3_rr/vendor/backbones/cs781/train_b.py': '3d0bac658a06799ece3b90ca23a53f6a2627749188f79418ef093344bdde0758', 'phase3_rr/vendor/backbones/cs781/train_c.py': '63e88403d9e6fbdd754b635d08f9e38d4b1898ff1373e927723e15e8274d0c4e', 'phase3_rr/vendor/backbones/legalspan/__init__.py': 'c24781d92781386918ca9056ca807b62493db4b5096d078dfcb0b1809b48b37e', 'phase3_rr/vendor/backbones/legalspan/bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/legalspan/data.py': '48b7af4ae83e715210f201777ae4c876876af550a9d1f09b485d29250ba7a06b', 'phase3_rr/vendor/backbones/legalspan/generate.py': '315c7067041e2314df5375a0d0b29899af0972366a21da0a546b43687d2748bd', 'phase3_rr/vendor/backbones/legalspan/model.py': '0423c932eaf22a919139ed433f1a5aee5e8a2a3720edf4e3b9d19ac5023435d0', 'phase3_rr/vendor/backbones/legalspan/pipeline.py': 'c20587337d8ac74b24ec517020d9752ea508f31e4b802e5219080f9728692cbb', 'phase3_rr/vendor/backbones/legalspan/train.py': '86a6d99aba0a153d0b0759662ba4257cd3e7955c0283a2550747f943b29017a2', 'phase3_rr/vendor/evaluate_phase3.py': '5b2babf1cb42a6f211c3264b7dc4be61566cd22df9eb2aae8b609ea0ee2f0b1f', 'phase3_rr/vendor/legal_d/__init__.py': 'fe77b5788e408a4d7c774eccd3cd8b359527cc580ce82eacc38fd5294e6c8989', 'phase3_rr/vendor/legal_d/model.py': '35c3ebc178de3a5797884542bb739d225a3ea7fd32af9774b759187c73c272fc', 'phase3_rr/vendor/SOURCES.md': 'b3cacaa420e9604392614d9e6628cc591b9d9c9f494b97b765b16d61fa12b3f4', 'phase3_rr/vendor/UnicodeData-13.0.0.txt': 'bdbffbbfc8ad4d3a6d01b5891510458f3d36f7170422af4ea2bed3211a73e8bb', 'phase3_rr/vendor/UnicodeData-14.0.0.txt': '36018e68657fdcb3485f636630ffe8c8532e01c977703d2803f5b89d6c5feafb', 'phase4_cjpe/__init__.py': 'f18c25fd56e59b628ab217396751039d3abbbf12ef6b0a935337a9c7b58c8576', 'phase4_cjpe/common.py': '0546d963cdf031283a87ce032076ffc832b7f49f15f6af1b0a9d4fc21589e069', 'phase4_cjpe/data.py': '95e6823bd24a6172ef3d7840112046fc818c7b069fa557638bca61adf9577703', 'phase4_cjpe/evaluate_phase4.py': '16f71e82d66b8b7a9e5289db22f3285edea99a4e6df8e9617b5b9e78699e3766', 'phase4_cjpe/export.py': '4a743510077a1fccd518774df56ecade58a5bcdac39fe69fd533408ad1f45090', 'phase4_cjpe/kaggle_runner.py': '8a8a4f82e083f080b93641eec1381b43720cb8e85582bb2a608e58aaf1eaed77', 'phase4_cjpe/models.py': '2a6fc936dd37531472824414c64bc0e561275f90c9448708e08625cbf46b9562', 'phase4_cjpe/online_teacher.py': 'b4ae71b323ced68019127241e991d87c1a6c0995d773f33a6efdd3c914e0aed4', 'phase4_cjpe/runtime.py': '066e8144daf6f5a1976cd52da1a0892955f6a7460e92a41480b28c65920b4469', 'phase4_cjpe/train.py': '2224e8e18e00f77ec52bda3b6c85a1de3428ec2236582791ca995137333248b9'}
EXPECTED_STUDY={'b1_study/__init__.py': '27aa66216e3a4a6db758b70d85bf6d2005adcceb790da04eb3c41190b56dc157', 'b1_study/data.py': '74f4e92d710de2d56fddf9c8917f8e8fea0c8eb081ba832f69db9227e0e378be', 'b1_study/model.py': 'e593a501d0d56934c0571bc9029fb4aaa47ab99a78c895e90a98756316417aa6', 'b1_study/run.py': '1a3ce9211e7657a6623c2766f95a4675828695a4e5057826167ba70eda28e1f4', 'b1_study/train.py': '2b644ff35086d0efadd614156c6d50f0f006856bbf6b4494b09c5645d1416168'}
matches=[]
for p in Path('/kaggle/input').rglob('SOURCE_MANIFEST.json'):
    try:
        if json.loads(p.read_text())==EXPECTED_VENDOR:matches.append(p)
    except (ValueError,OSError):pass
assert len(matches)==1,'Attach exact recovered course runner dataset.'
CODE=Path('/kaggle/working/b1_study_source');CODE.mkdir(parents=True,exist_ok=True)
for relative,wanted in EXPECTED_VENDOR.items():
    data=(matches[0].parent/relative).read_bytes()
    assert hashlib.sha256(data).hexdigest()==wanted,'Changed vendor source:'+relative
    dest=CODE/relative;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
raw=base64.b64decode('UEsDBBQAAAAIAKkMWV2KRSF7PgAAADwAAAAUAAAAYjFfc3R1ZHkvX19pbml0X18ucHlTUlLyL89TCMhILE5VMFRwMtRRKM7MS89J1c3NT0nNUcjITC1KLErOyExOzFFw9gpwVSguKU2p1FNSUuICAFBLAwQUAAAACACpDFldKwJLRJ8MAACxJAAAEAAAAGIxX3N0dWR5L2RhdGEucHmVGmtz2zbyu3/F1p0ryYZmJLdJe76wmbR17zJtk47jttfRcDgQCUmoKJABQNtqmv9+swtABGXJyemDIwGLfWNfiNh0rTLwp27libDf9dtGGP7FyUK1G+iYWTViDm7vF2ZWJx5Q9ptuC0yD7E4c9Ipp/mVZ/dnxrGo3m1b6g8y0G1GVSCeFWiy5NinoFUtBG2Z6ff98zQzzp4Ws+V2JKylUq16uNZJVfMEVlxUv7VqA44tSqRGG63bNpfiLq5OTk5ovQHeNMKWodVzPU1iIO16Xq7ap2x754rzOv/p6mlycAACcnp7+yHkHnRKtghvWiJoZ0Up4+b0GJmtYqrbvgL/tWQOyVRvWiL94DUYxIYVcguF3Rmenp6eE7laYlVdyVrVS8srE9TxBkapWWpr4Ue2thhzXMn7Hq97wOHpz+dPld9cg6rRhc96kiLrUKwY/XL3+GSqmuYbf/3N5dWklzE+JiVN4ffX95RV8+weIOkqyBTfVijVNnOyItWbFFeTwTl/ATM0mBSxaBQqEPMLAMYrPR7RSiHWaJMWOzu6D6DWij6Oa30QpRIZrEyXvCXS+LUWN7CArF6AGblArFsbZC3LQ3MQjG1q5xAIaLmO/Bp/k8GR6Dq0C2Zrd8We5pZat+VbHSaB/JjSH31jT80ulWhVHl3cdrwyadsWhVWIpJGsI59grIkuf/AJN+M4yjDKs+TYFazvyi3LF9MrLNdC2RzPNTc0XrG9MPAC3CuKIbzqzvYjgEWJMUpgVSca6jss6jgcayU4RTG5jVMY7WidWSgeE1Inee1LRlDatT/sdnd2gGnScPKiflzWXRlQPXANYWS9fNKIyuEgceIUprrm6QTMyveKkOTLNbF3Mzot7gidWo8ilM6bVs5JLyEF2mWKybjeZU2Gp5DLGuw2PYPqVpVixRsyVNVwOs2J/ceAEfcyeQaI7xcWTFHygsGdljZ5gz7TK8Dpe0ZFVCstAocLwjY4TtM2K/BF9YE9+Ci6zSTGbFpDnzqQ7Us69B4oJPIPzJ08HZo5YSep+sRCV4NJALfSfrZBmpAnLojOKF1kQh3KZdVxtekOQ8R4Dyezi/MnTYszBCoPYDmYmpInFXkQIiHsvtjzMVgXKPykGXg5bKGN1Ha8sFL+rmr7mGD/2Nfr3gaN0hrx0MBlp3QXBjzCcp7hz477BuFSLysSEOae/qffT3HuGC01pyJbfDJaSFD7/nEK0pTDcD33x0XfEsjXThb0lB0MxsYnB2HGGXwM+fHjGEyyFOR06cCpJQ1wjBLhzDHlwj8SCbpzjmRUJfBYuzIsERbV6mLECPvPf58WH/P/aBaTHQ9B+HHp/w9maLfkuJpleeeWl8C4aQlvpQ1vpLswFXUj7I0khaumasWYAFLWHwnifpPfzIgBEIXjZa167M7SOiAfOCaTzbh5dwLXq+RG0oetXbS8N8jI7f/I0xbBRpBAR66WPCQ9j8z5f1n3XiIoZflRMOAvYhzOYTs6/fO8qsVsh6/a2NFig6dj+k8KG3bml/Mn0PAViN58+TaG94aphXT6dTFLo2kZU2zwyTDSR8555Xy853r4BB5zBuU+EbvsZTJ/6SmCCNYDDC888RKssVQyqDyY9SdZwgkAtNlxq0UofPzGqkPTEyi4hI+Gw2LNuNptNU/gihfOiSGE2yQr0uEXfNE6S6AImKUS9FG97Pl5T3DAh0SuH1YOWu+cWKDpb8nKhWEUX8QKmWQqRlQgxTe2110aJmkPuVXTmteZ2mTIYlBqhTayYXHLMjht2F09TkHDmjz2CaZI6ZK5C2Z11sQ+vult7BDNEMQlRJMVQ2aBuLWgC31iTDWrtWi1QJEQtu4xZroIjQ175FH7nYrnC8u5tz6QRDWbgpmnRtWHTKu5MrMG0VAEuqPwzK6HqfwFrbtkWmxK0AsxbswIua52FEc26K6Zy67HjSHVL5B2ntyuueDyw/00+EvRzOIfH5ChZCtNsLzv2m75hRtygqWSXVf1G95vY4d+DXbWao4vKLtOcqWrls88OSYp7jZC6YxUP1meTIg1ozc6m+Bv1H2iVN2O5eykWrdrsiR5yobBACClO0pHoZzDdkcmYNtuOx0KakKTmH8oBv8q1bG+lv7SaN5xc37EalD52i4oJdMoN65Ba6lhOEvgb3t3n8P390uljRQudcyTgWCbv+Y69BD1kz/n9Z644W49W/SmqmmxBFkjs7+LMfpuJYpBidz8dWVvHCWm4umENnYoxfgsZyxTw9robmwxlhkXram2MIKTdCf3uFL8Rba9LLodFOomHUrxTiGJHMKi8HaZHFPlRtQh7Rj/c4RB5IPAeTTwQLhEiF6b42x4nDiQnVuWPwEbbGbuY4y8swcLiaMenlTYMRzMGj8PgOLUaYimUB04yrbkyzuQ+NiYJtQWBBz1zTkCtgzPfhJqHSYhmhxwvresuJJ2hXOhV+SwHGRZBO/HTQZL7CUoeSFAO40ckpAN5DIPXHM6AHVZt8jFYDyU5L+ZjYnnIdlbJTtQEC5WqYVrDG9MqF1mwcilLIYUpS7wMi5RGZSmIWgcXFXcy3MDoYVSMX0ehZZGJGp1B1Hq8XFFDuj8msud3HKj21hGn2UtKs4CBOvXsFLcswntTnI5tm5a5UY4rlcaznBevvgdR589pkDPQcCOkVvJghCQWjqLQ8KqVB2Pwj3xrI/AI234VhGPCDDnTMWGk/i+UWodiBxK7CQsFGafcGcHsNQVbwZvagpAOB14GMlXTak509g2KqrS7iStiO8U7pjiNKGMcb4hlCvNe1g13h1XbYk2KA1QHMItUL0tcj1x3i9+zzboW6CiKS6Nzqr+B3wltynZNPy3sbavW+/hwbYQQZ5usrwVSHmaoO3j84eBTi/Bx0LPQgQiDZrtUXGuCzPGPG4PgwA3y0FgjbkYDuahIMsVZXeIQKPbFW62dCUvP5YG5LCZ7ixFnN160RYfdL4kQXVghkWTDNQoT2QbEDppjvJM4X/QzYAwpK2a5dUZCyXf7GYoUjcJKtGlrTiHDmY5TzyXtGIRIDu1GABcsFqMY4wHs6LrsuCpxlhoVI6quvA7g/cqArXQlywAzXkdI3faq4k7uskQ9laXr5StWrTBKOAeIrR50ieMDp8BFl8wupl871dOBj/XTmib9G6bWNF62xLyydWYDXJQOG1W76RpuuDWCL/IRTUbocfSCicqi3K0FJb9L3WPXdOD3fNDh94fQh+SSqw5LtajAeeii+1Ax+R3xHhyEjdAbZqpVUEf6BAR5QMwvRjbJjytX0+IN371cfNBfh4N80x3Q9bwXTS3k0is9lB+PHNClx5b1shFyHSfYJr2WzRa7Hg1LLrliWBwL6e3m3COQ+1Aiq+fB04Od4e9BINkRzJC9vru6fHF9Cdcvvv3p0qUtG8Lh+vK/16mo7b8+u9GPX65e/vzi6g/48fIPF+5FnSSH7ePG9SS82o61QaXoLl1QOSvq3VDwfu2Nz1vajwLptSSfpBAUTPhzVC3hwl4NhEsudOSTcf9GyqGXCMCukkZuNDwjfdBM79i0jdzNAt5DSfU+iUjjTdlvyND4tBDWN2M/uTP+ReHwqxVBWICjj0dhwWHleqDiCD9icTQ4U+O5ey3caz1HxjpY35Y0Rx6/NdJzTIrOR7OzFL5O4evpP88P84afT+GHvmmsuw7OJtCDDL6atJI1zdYNlBdcYWGKVsBBQ8OXrNqC4ktsUTXDm6aGK7b/qby//b9uhoU2Fr932IqeE/k7W0v5Ynjww3GRfFju+534h7WNzN+bCWZcVm3NSes4LD+UXR9sAo6n20OZ9UgiPSykam+9vkWdD29vuXvlsyTzQ9LmodzOJfLqMJlRBHz56s3l1TW8fHX92tftv7346dfLN/Hz9Hn6PBmX66lNhHW/6TQOZJMj5qJYNYvoWkYFNtLTg3C7F4V41Pnda/sODCWDGuhIGBkYWRML1WxdHLvwsaBZIvwDppMJNbnHcaL+8P8kCP+Md4xwjzpqzVAwCrmMXJ2aO6XaQE7UMQgY1tB9oOh4ID4/RNor/V5/WlDPSJtjtRbD3MDth1YokgdD4SejUGhTADZr9/jyDLn+yTMzgnPThJFr/nL14t8/v4C3vajWZbXi1ToKYzdaKY7adZQGr4s4R232Ei0pzTVZY76G5mtUoijeNThTw1Jx2Ar+44mrAVMXG4dyLV904f1zX5whQwTUxD2GiFSibdllW/6joNRjUXviwYm677Jy2hp1QXnw/QEeDJq7tMVWiPmDctlJRY7jCFKVo+Fab5px0IYVzGFFMSyvQZe1a9byUaMVcIYcfHg4c0T4QzY5+R9QSwMEFAAAAAgAqQxZXTseOf+9DAAACikAABEAAABiMV9zdHVkeS9tb2RlbC5webVaS3PcNhK+61cgzoGgTdGa8SMu7TJVycbOJUltJdm9TKlYGBKcgUUCFABakl3e377VeBEYUn5sbXSwZ4DuRqPxoV8YNoxCatSI8f6sk2JALdGk6YlSVCFmJyUde9LQszP3XQvZHC21+ejpOI8GS879eDfxRjPBSY+IQm9iokmzXpXNkTbXo2Bce5Z55MySj0ei6LNaynIQLe2Dbr0gbb0nzfVecBqRPq+btyM9IX79jrWUN/QfRNFfYebs7MzsFf1MNG1f80a0VP7Yi+Yac17+Ktqpp/nlGUIItbRDdc0403WNFe27Au2B0E3Dn5pGKnFeBrJ8nqJ9Vxp6VBljly2lI3zAVkpKSbSmHExWH4imqEKcl/8kkgxUU4mt6d5TKRTGeX7C23VfyBW21Ql5S2TrdnVXoHekZ209EBXv7g5V6A49WVXwcbRBmOP45HvNhRzwXZ7IDqIl1ZPks/SwhURu1yVigchJzc/OzmAjBylua2pPEXtQuC2wDvWUh9HSkeXomwq9nHcpCVMU/Zv0E30tpZA4+4mO+ojo3Ui4YoIjSW8mJqlC+kiRkOzAANiK3SEn0uJCZXZ/PbmnUhVoIOPI+AFVaHdVoN2VmeyERMwBCTGOKJ8GKommSz1nFa3Ekowj5e0pfNwyfvZDxultzXhL77JLYwDLnaNztClQ5vUPJKxAGZi+rQGc2SV6Q3pFP84LsA4xUBVvC/Qi0mqp2fJOWV1nWX+Jwn/Kyet7akR7Jey1/oUp7WWnxE0Hp+Sc3nwQTXcoEPfwqi1r9coyOwS7zTg4Gsy09TSOVNaAXi+rQHaTnMrq2cV3W2dGTgaqDEDMV03kgWozgDO3bFYYvOQBOZ9BzdXufHuZinti5LX0q+Q58lgecHRkYP19gcxhRDh2a83o8K7PXl1H1yGmGFeacLAzjBVLR5wj2itqGYO4ruNBWNfxMC76tkC3rNVHVAFRebspbyk7HHWpjmSkMYrDEaC/V8CYQnnpB968+S3yAsOkNGK8kZQoiqwcs7C79ta3vmMNLVCr70e6UMjPnowC7bxPJpW2qP2FcUokNmtE+CnQnhFVmVtauBWrZOHK/DtrpY9Mtv9nmYo2gqdCI2lO/tcKvWX66HMJUR8kafGJvzHmcZbbXYq+vSrBCdQ4sWnqb8z2H+R5tspjt+ePCEJoHNsXFLvLAi0kbxeSrZIOAFv3/zPACWyrcDILq3HgMk7Ce8su+2Cv4Mfyg7mDH+E6XH4Qffvx/PsP4Qw+ZombMjJC7vOTaKaBcv3FeY85z2r74mWBjpS0qnoOHgCc2XcvXxWolWIUk67KzVfkRkYQ7LnvSicAVVZ8kPhQjgRZgGP1SYGDIvjo3yBHMCrnxdpgKvXm+t3a3XiGHqM1cjHp0Wi64Fijnkan5pyfpUzGik7NlYlUWituwQvOtPF1ezi9iw5lX0CEPQR1IcM7cZQ3BbouENhkR0pJzSSO+OLDszKePo0G81JLwtUoFMWbAm3TW+MvgpCIQNjwJ4BXDhjSvLI5TvwaPyvQ+Sa3UQj+LFbelKohPW3rVuh6lKKdGl0HGdhtJICpHqsEbKyzy2tJGIdEzYSdizJfJMBvPA+OIIDFpE/3umYvezh5ivQiXXpe8j2qUlzju9Ws+UQlQAZ+UyrWTzjgDb/Pc59QTyN8+4wS3j/8uPlDT+09XlRPn6iMQqLTCN6xQ4Q5D7Q4ryrb2hRqD3uLWWC232Trt0uBlrVd0FRZ8CElgVXC1C4DiJqVs6uUzuCs3hPdHCPyaPSUoYUioZ5z/PVKBEBmhZUHqkNSVxvurEAvc1RV6JWF3u5kCYDAvMCDueVyEUvZdTwrbDafLxbw4Df2+aZCmaQdlXDY2UnEFdJED1MBZKMQfVYgY0f4fxSKwWWDz9ZYjeCa3mkzYD/C1XxLTT8gK9a8gf0L5OZSx/xwDeB768JWPbC77CQxsLDsidbSIRKUPg3ZfVfOyqy54+2Llys8YWG/3YR1a9jibOdTImypuKhLdmlMtul5DUaXhB8o3uZX+fLQDnKqBe/vM3u+qchPaXGQk1Xh59//hY32m+0r2ELLpLUO6Suoq+Bm6+ZYmwzFjKyoYbxgJ+RAZazOb9CgeViHtaj9GfsH5275/qA3E3wnPY6OA3bz8rmNqX8SfsRJeH35vECbk4rUegBwfaxjvmg8EZ6oaAT+5BxwuXmRrGAU2CbtFkr0JKl1JA6eDOKnaY1ESAZ72IaH72+FGToIee84UIUGV/DhSE4St4ZSi+uaDnvatowfFBDOFN+i3yE1RSMxs6ghnAsob7p+Aifg1nFFIPK4hxaIFNPhiBoyKWiC0L47D6dSzpmucE0RQPDgi8n0yi7iL+EtGkx0gNSfwYHPXUHGD8sbDxudSXBPhn1LECkgz9n315VR4RI+YjtoDbW7uDJdL2/USdFaUsoBx3rtCgOc11c3K+BIVlghkL+DkhJuvN2tuKYusge7+EBfdr0gGkfoNC6QQ7Jj5JZNLzjFi3moPy6uICUC3ee+kUlBVOBW04DPNwtuSKO0prKGJAZ7pnO0mfOZ8w3cGh9NkqJa6CCH8HuQX5K+Py3cltX1a9IckYkZiFPaKkTCbo2NorI6vvaREVUjJMXWvHmpbiZK34OqwY4lNB1pW3es7/F/nPQCndsSk/EuL5Xo9EDuErMMlMBKTjJ6HFv5sjBu7So3xtzk6OlsRTOw8xRR985U9Fbazq5NrFeHbhfcy1Aes1a5NkHuM0g4iW2UGUzDQCQzgLKylCbNNcagdREpHYy2pnZhtMoLdLHILvG8wuPY2gO7C9a6CNtM5F48WIOYc1bF7Ej8jiOYeDAZUiSk6QXab6ZnC18D/2fh9Q9gfBoC9sDUYNK42Af+ODXX0CCDEySM+6Xta8WRKdQQaF1IqrSQtEVaIB+KkJCm8Us7ISkih4OkB5L6QEtRISWkpi2eT9ztKS/QNb2vnNNitvdp53bsKnICkqqpt21BY2z0ODZN4nNF1ymq57ThwqAIG1V89h8ltSdWVLSnjaYQXwzHzoq7dFKfLPhnWM7uBmIPucPpZmy30VV8Zo3UwbJ2xrN7sAABgdhfhtx3jixpL/jhpLWUyvXhMpJc9+zahc1Y1F6IPuUFld8WVuu5QRpUWsYDRW/CUkTVmnIl5GyEr1TdmWX3tkCX1hY3Obh4RW8WdGabC0rI1hLSb5Hrs5pQ64O7y0hJLylp76N4apq80HWiXPf3JfrBNahOZEYMRIqJt+adBHyPpD71QSPRR3Qrpr6FPjsl8MbYiGGc9MmtMZa3TCFmpvnTWsbzP52Xu1c7Bsbya+7ezqA+nvhYxxDiSyAchVo5+djZmZM/2mbv6bGfRNlVqQ3RGAQWaIPOYSY3LZH1eq76RD0Hezr6q+w1BMmpMaNWLJm0aIjS2Kpb286t+Vza7VBO9j1tXea0NLQr5ApU+xNNakV8tH5tmf7ESlu+qCTEYRtJ5eij1rL6xG5odxF719UEA+pcHFn+WMw2n9OLOOWYQ2PqUh8K1/h4GqOTQFqgYwkRHV9AnDgRGoJQtRKt4yp5JWD7FSD40LFlgy3q0mhqM4HQ1MNe5kwWTiQy7zH3B7IomFOAJdXBWnm8hKzNsx1QkvTaIz+pbZkyuQRQX/51uA5GiNdeoHl3cRXru2SyYI6eQcyzQfB+y+oXH78Yg/FRzhWuRZ9b5zPI8E/tiuralGq10uRAsenfFch8cbZ59OjRD32PxKjZwN5TCU2xaVSI3jGlbVqlNJH6b6iToLRxDFL0CkHXAPmqD8Ebm4kKjx49MoLNWqV/hzcvQzWecWunQ3/xhC6qWdKyNOVyrbkkH/2S18oUFvaF8kFNAaxgMPR9hTYz5/xavdtbJR9W0OfJsW5res2Ia8RkfMV2Xr6q0MY2aZ4ng1s7CFmEf1Bev7VudnduhF9eoSeoh6bTic7+FTnQfZW1Vs7WN1DhznwVY1xqP8zorsvOdDx967NA43wcMNDWo/+RjcKmHzamIq/cndlPrHedbryfeNuf9sfnLrdrQFLXckdV+kOnwA7d8Aj1qArd+kUL3lB5geU0tpAPaaKua3Dq1UpP3L/0OWWSHndl95+Mrbdzo761Y4pGivALGmsX176vvJ67LB7OrtaXsC1u2qYy3hGpkp9v5AWaT6oC7zaWkBn2rsEanWt8oi47eMDjXaQ/ALGDXn138MEF1tYFet7k8L0hzEP88mdCXwC9+Tbh8LskcA9koJBnSK0g3uEs2CTLTeqfuV9UZdaT2Pfh2dBAAhNGDuWtlYIz8xOtrMjdcwKnt7l/lx7T+9OyRmOjqarmbfYygM5rVPcSYAePPoz09UMUeQqDWDynt4lkA+cHpYbZ/Orsv1BLAwQUAAAACACpDFld5PJ6TJQGAAAgEQAADwAAAGIxX3N0dWR5L3J1bi5weZVYS3PjNhK+61d01R5AZmh6ZiuHrdnlwRNrMt6MHyXL2UqpVCiIbEqwSYABQNvaX7/VAElRryTLg0QB6O6v3w3JutHGgTDrRhiLExl+P1ut+ndt+zfbrhqjc7S7le3w6mQ9kDsjclyJ/GVSGl1DI9ymkivodh+E20y6nY2w+CPPnxtMc13XWvWHhNO1zDkBSUDpt8lkUmAJb9q8oImIY/x5AgDg+aTOCDnQ+h/c4rpG5fyhXKtSriHziqWVFoWNCEZglBoUBXf47qI49ueN1g4yjzQKtAtmWsVpnS3DGWe2AYEnQNtWRLInu6MN5+kZaRV5GZfAnvWKB/KU1lnSMQtU+J5j42Dqv6RWICygMdqMZefaFJBBIXMX+U3utg1m9BF+xynnStTIeQI1WivWmFlnus1k57BseEtLbWrhOL7nUZxAKWSFBW9dnin91tvpnE50ujW4U4gQ7kgaI5WL2H/uZ79MZ/zr1c336TUw+BDcU7R1YyOvDGHOegfQD7ZM4IcfOn4Eq2rtJpubFnfcjZAWu3ipRKvy3oc2Ad26pnXej1389DGjTb4JCyUo3S2keVuIVFouXoWsxKrCKAZtxpsFvsocea5b5aIY/gUVql5eJ2IABbNWUZ5MyewRm+HvLVqHBUirK0EvPz88WRAGPYRBKDuOyrEiw2ZavxSSssOgctabJQF8l9Zx/TKy0rNeWchgsQwZpA1IVeB70ueJVICqrdEIhyd0KXVVoPmz/Nid/KuwutDQxLqTcQlMvyk0diObEE3DSVmGw6nnZKMRQHo82X7Gh+NHyd4/VICEKnq92CUVOxbDJVCqeH4L1siCLWmN5XVRSYU7QB2ojs0AC4gncdipRCkfzBZ0ilNUuS4ouKQaGHigq63DI93OBdR8Iy3gu8gduRhERSy2sBEWBFTyFYNV2Mjcwm32rH0I7WSaE1UfLDteqF4hA21TVK/SaJXmutlGe/tp2xQUUz89XV/xX28eb758n/Lr6a83P00ffUHycRgn8PDb/Nv93dPdl6evX6ez6XXGPrEE7m8f+N3TLZ9/m02vrh8z9neWwO0v3w8X92QuWGD2cDX/xpaQeV94D3Neygo5j9MuNBefljF8IBVIQ4sNfPCo1+iiMZcE2EhGpamzjNwbmlRa6TWLU92giphgCXgfS7XOWOvKi3+MvRC6KmEbWmz64CkXdmtTfMe8dVQJEmAXLcm/qOlz9Ylb1xbb1LRUZ9nFxbNescSr6FvbMjkKnBOPdYVuXVbpNZEWaEw2AvI4v75/mhP+1wzVKx0RxnGFb9yitVKrgxzeCxXKuSS0pkYWWc+0kUUCzWZrZS4qvm7arCtBnvm40/SBlg1Kxec6yeP8ajb/f1rJKQAHKM91GaqhqWgaVEUQ0NH0tAnFRTBphz98JaDL0qLLPnZqEAep/HCiV3bxORTlt42ssN8bl14D0mFNdaLbXHxe7leH/fJMpxcsyGbL02XaC5Ru88dBfBi/SRhEbMYMNpXIkcU0nVhnUNTHBSuspxbxpQMVDHEIxKeUVOjTIdBQGfNLo2rSP/usdjQOq+rgOBmP2JDxPLtjkH+Dr9q8CVOAyF0rKkrNtaHclMppaNpVJe0Gi0tfTUPZIEfb9IhVCM1D84fIgw/AfJQSjNRYZ2QTnQk0eqg5QNbp2gUYW6aNPtLRN6ACQVo/QtxphcdaBj7k2mWaV9riCbs6NLVUohp3h+hoWB3kZRl8BKwsHgx/x4yHUXnUmXthe82ZeA8bQzf1MnaTbsZupbWUPwPgMB1SeL5LxwldRh/HSE5MrhGbT2e3N3dX3zm557z3Ou32h/Vj97N/33/hPcvzdem0nEMNztVy64TDrHOL71V+hRpCGNsJp3XY7J3RjZO1/C8aHpqyZfG5WjfoFCpOarDWr+hjO96bxw7LlQ8kWWNqK8Qm+vQxHLdtXQuzPRhBu7JGVfDcrPlXill5NM+MA3Y41mHoS/goGMujKCxPhd8fOi64ZDB/QHfqpvQ4/fl2ejfnj0+3t1ez3/obk5dAlsg6nAmUUvnSM7qDBbayBKG2kRm7PvbZLxVELNd1U2EIh0a0lvB4g9PU35vhT24q9wrp2lNrE0bIIlwhgnr/hMagRUMFsa+PUEixVto6mVtYYUmEBhsh++HToGvNIH4ymcgS+jsqFRPGeU03ac4CMv/3BMVB/1dFemXWLV2yH/xOV8HCsVQUBRfdfjRMRQZ/b6XBYhTdw/8Jnsx/EaGN4vRZr+LJ/wBQSwMEFAAAAAgAqQxZXcS5W1+uFgAAFkwAABEAAABiMV9zdHVkeS90cmFpbi5webU8a4/cNpLf51fwEhwk2Rp5xk6CbDYyzpfYSXDZZJBN9nDbNxDYErubaYnUklTPtBf33w9VfIhSq2fGSbYB25JIFslivato3vVSGVLL/njB7fO29k+/ain8c0fNzj8rKhrZ+Tc9rHsla6Z1+HIMj4Z3zD+/5/2Gt+xio2RHemp2LV8T13YD4H1HMXT9kVBNRB/gSFXv7Eh8LITwQzeDqA2XgrYw5J3tpPcto0oUHTOK19r3pQem6JZVvWI111yKStdSsZzULdWab3hNAVSlGHTPiZJ1RYfa9rpwK99RzT6p6l97VtSy62RYCTWy43UFaMtJw7dMm5zsqGruKMwh5F1OND2wnGjGmoq2bU70juZEG2oGfQqe3SNcD34wsqYAs1dyTde85YYznRO3B81aVpvK7BTTO9k2OTnQljfUsOo973Nyp7jBrTccEeYmLBpqqJ+jV6ynilXwzTV3smGtb18PvG0q/JQT2Rve8fdMVVslhx5WwkxlFOWi0oZuWU6+oYY1b0UtG6b+s5X1/uLiomEbouWgalZZJKXZFxeEEKKkNKREWkirCmilqrIC1iOMXl3f2k7MDEqQf/aFYi01/MAqI1MYmhVUV73U/D7NvgDEpn1GNlKRntZ7umWEC5Im6+tKm6E5JjlJENevKqXCi0V8kuFU/ocwYLSWyrAmxdnICw83K9S2les0eZZk04F8Q/qCa9xKmhEqGpJUVX+sab1jVZUQIQ3A7WGTRmOHvtDDZsPv7WqLHhdamHuTZP/nkFfT3gyKVUpsA+osVuBk0/5odlKUlkuLLTNAXizNcstYpeiLsa0KjchXpeUuaFBiGxonu4JfPTTU9YXH6QCg7DSD3Uc9uK7ogfKWrhEXrNWMrG4ztyXFtJFuSwjD78suVPtN4N+rxG4xubXoHjcE9DfpiDv2/exq9GRvrh82JbdF3Q+pO8SHlm8XN8LELvoEByttASIFaaQgOx30T27D7p2USlu6Zq3OSSu33OicBFYui0/DQeuhBS5Bpj8/wu4BBIUm5VRgpLaz78EaUpLU9nxdRhAKqs2xZykXJovmXiUoLO+4ZsktKZdFZ1gYwM+JfStXVzm5vs2JHEw/mAqotfxZDSwn75mSVcMPKJTLq+l8TgrjbJtWUpNO5HI0l1zrbDq2V7OhZ1TAGSDIVRbWxcXFfzjdI6utok2a4eE5gZo6oYh0nBPdt9zkpGEHXjNQJNK4E8R+BTvQNrWz7NlRo6weQJjbdZCSrG5z9wd7aUNB+JASdWrRSSGNFLx2QIDCOMxzB2TGxNAxZelbKlYoeadTXFIW0e4dN7ugVVK71KgZfpZUSGlXnSp5t0rq3SD2OrnF2VYJSFxUJ/DFASksrrPVlV284ycQdxaFXG+44IZ5WixQZEznVpRrRt4BJC62N5IL81YpqdLkBynscMLFhikm6lhkAz4L2vdMNHbBvPESAH4W0b6Dnd+yaYHSwvM/bh8PYwIMP8Xw+IaknDwn1xn5d3J9dUXKklxNd2L1O6qNnCRw9APuKcnJ+FLh+ZSOcNx31lQ11UyXOAMIaUNb96llwh0vb/QKx91OCddSlugLbWi9T+3OM09jOUnnpEQuPZ1l5AXp6H16nROYByBlXlzJHmgLTS5go03Lt7s5/S8Q/kcfffSm75U8sIa0dBD1jtC6Zr2homak3rF6r/9MhCT6KMyOGV6PFgYZejBkio8++miZiYDCgWkCvacJGiJJTvZW/u6t/PX4cs23qy9e3ToO24GFUzoKNUxoqdLV6jonn+Tk05x8lpOXt4HGS0fqOBQNrTBUCqarlu9ZiiBz0oAcdepyLWXrRC9tGmTod0VPG983vcrJqyzuUXno9rUQLHUCEhl4JpNGyvuYvLt59ZLsmGJEg1lHDdMIhIstqaVSrDaCaU3Q0Gvl3WUQimTPlGAtUXIQ0L0IUGlO1l4eFBtG0RJZU1Pv/A5wuVm+2MVuIZ/sLFsWEbRt61ZqlsKMORjXbfmSXX6SE+UfFwXGT4MAsnai4sYiGSyrQTiMowAjbmGaNFzTrWKxCOEbt/qG9WZXdbTvudhOZ+tYJ9UxoGJN6/1aClYYua9Yt2aIZm2RcmpPrsEYBoqcjWbWVp5OFeGGay40ckyKIBZM7BlWTtaL41L76k/rSTtza6uEVJ0bPx1IazPQ9tzAKX3EquGBs0eIuVuSo4JrdvmZowJ4XNjvEiX8nSl5uaWGEXbfU4FUXu+o2LIGdDiKA9YQt0lSy64fDIo5Rxgfk69ARhFKFKMtSEfeDd1lryTY9wSEsuVI2PcdVQ24fIZQYrg4kkGgv8IaYuSeCaLZPwZQXJazQG+XKHBBkOUgucuWduuGEvUF0UOXdrRPWyZyokYNnDk5YdGNO3Di8BHVbg2w36zWR/0oNUjdd0WtpNYVE0bJ/pha8PlMksbq80SMBpvbji30jvaM/FtJQAO9zIhUy9aD1tG2lo79O2HFpzdTmXJTuFMFEIU/Mbc1kKSoTfoCHkcH0CIaRGnHDFPaejngiv5j4IpplMHOjcNHrnHZP0jBLLk7MrczuE2B7QNqIZ3tbuvMIqu/trAAHBewQcXRDtwWdK3TrABCychrcnUy5GEk/YVrDUoB4XrbylC9x9Ec/G+HLr7BWU+3N0dSYP45tmDADGOPrO6dku+ZIAEQGQS771ltWNMeiWI142BTzJdq14GeBapGDE7ISkjB0OuwvSC4gSjybgk4U1ovWxmPGOEP62M00Q3rAEfA5VOx9QRjPCwpBTAxx9r3JZZddFSPot4pKfh75vfFaimQ5B8xCcEWhLV7M7OWCrBh4w7Wsqi4OFDFqfC+nSWnCsip8mfkm/Bkq5FEKsW0PVnbY0GT2QMFF5s3TBhujtWBKeDspkRmWNDcWT56CpXbbNUzhZZ06T4sTdYzuq+2/VCtj4bpOOLR0fvKaiVw9WUNoh3CKE6s+bDfUvAE/XIMb6Eq0CXI/fMyHglHearJ8jB6dAOwwZ5JFIRMMZKGFjh5QZJvbn6pbn56++7777759ucCOiS5O0IfC+DCpNN+JCHPMRBcNEPX69T1z8mmHfQu4qPgK0O7cxRs/PKMZ1CDoNlaD8GGQEMMkzWVNqzPrVPQg+dX7ajeYbetZ4uNbEFRlzZu+IIkdjqdkBckTTTbdkyYCtavjUrtdCv/HRAbQSm6fcNV6mKNjjrZPdemkvtol1NffYtR1goDLEtBgKSmLV9bXymZ+UQ2IuWDLdUY15lFcVM7ifPdpqNADs0grJLwJbEa52PyDroiT8JCSMN0rfia6QiQNjY8dPwz4YY0klnBXu+k1IxQIc2OqbF/EVA3LtTGLEIwAhqRZsHCSmNTnYuG1wy6i76gth2IGDaagRP9CbjQMD4SnZOZvEN+gisAqKlS9Gihrdxctx6B4UM2QdQp6zjiejE5Q880eNR2cgwgoeOeNFz/CqRK0BAjk7M/leM25FfOzy93x9NjZHu273L27kgYHHx6YO8rsFkV05o1y+uvJnHAQvTvk5zwRpcR3tDT9wHFchqVivvZ03Iq9GD5omEHP6BhB0u4YPOK+nieRxp2WOSNhh0qn74pQ4g0wI3mmgU9o4GrhJqqYRs6tBGZYDTwIYDFp06fzzMmEVphlrElUMYCKmbLW6YyAGfUYHYe0gpxxZtyb+VMCbL56LTBPidHYK73vE/jKfFQHDW7CJJUD6VUYOo48YFi1IWeKtsAKYhZ6Dlk/Ao1iHSlj7pg96weDETH4WRVGmbPcpJcXuLWEtt0dtOLqjL6JZeXI8YXgc0PJLvNQ7bEWndOtBt2bx8triIwcrPhNadt5SNzUtjsS2GpAUamFhWFNo0cDHlOkv8VoGfGz0yp3LqSXGzLZDCby89HC9r1syoT3M2H7d8f3ZKiI1XsV9STcOiesdDye4pEsJh6VBI4whqFQUTTp/LAU9+SnqwA4dpUj8oC6LUoDB7eEAL/jbLtcU4/5fLp7k7EjwYmCePT5Jeb739883V1Y/kMCMUbI4J2kER5ThJnp0Ca8jlJive8d0dpGGReKcZjAHQBjkJlc4SQH6TKe7LoQbgke/F33r+DrGMYnpPkLsnH9u9uqq/fvvv+zc9vv84gcU5VveOHiBLdB0v2D6Lj5NNs5YVifUtrlsL6bZuN9pJykqLG9nwSowV6uF1yNiyAnAAGyyk6cxBQHXi0UpQgI3DaPNYLZfS8JHTwIEqXw57YpOXMQg2vkamqd/Tlp5+Vc+N1YZ4xg27j27p0CUIADjsBcm8ZwGe9rHehHd9wq4wJm3IH18t6BB4GNOGX5HZp8gc8IkRxww6l41jLv2XMxNkf4y6Bc6h3rKkGU5dC3sG3HW8aJqqW0YaptaSqsTm6EuIMDylRJ1cXXJu4v3cXYCPaVNZreMAfAlb56e2br//nd/lCrizCeh/OG1moe/CkrAZRwfdA/VKapzopvrok8nlYMHKtbA25Cvua2ox0UFBjqO3B5PeSsvrm5hfigjtNMs+7i6FDM4w2On1pG9eDaFo237796tfs1AMdGg6IiotU3IjcwYkiP5BDgwgBbjaqW0ltT++ATiKo0gdeonUjBhSD5SN5OyoGJtMufMK7zto+pCT/3H9BDt5IO4CRZucpIEbjQmB7X/gR3FSQokC9HbOnDq+eAkByS7V3L8smkkcY2NPU0DBww+9ZU4FykgN+8G7xTg5K2w8oKCsmmmoQ/B7KTADipg+iFtdfhk3mBKYo8TRWOB0KInCCKlC4riF8wFas99HlrO5nvhtL2qU/uFViPziJCoAwYgISM+oUvoV+55neHYbj9pgEFkUEF4xu2cTv23CxZQplQ7npRyoLC3KEavHg7KExkRmShPCleNPQ7r/TeSGVN4vc8nJyxyDFWjWspseyuLp2bFHTNoJIu774RtHmr/h5wtKoEfx5ohq7ygnqEHjwcUN4HpUGvK1BQG6uy8vrIidr2qBRDC2nVKj3vO9BjjtFdpVbkYeZbatfyqtirk1HiW6lsZ0yiunYz0UPxOu/rd2X4JV0J/Ij5qVb5Dl38Ftm0kmrK0WChcTmeRdZQjUVDRopsC473WRloWNPj62kYyIYXtIwOicd7SvUfYDspO6HxB+trqRoj+U72uooZgu5BQtylURkl9xCYmTTL2U//wZGqRPFP9m1OiJ+YTHwwrIg6bjuICUbZT2tEITpXA2TC+u6FWAzMjPYTbH7NKHw8wBCl7h8whLx+UG2fTLCEfPYBT64aBf8fHKgwmoJzKuZXVEz3mKc6bQGAMKjnnJoXQ/dAJWFWGBGnoUWMGGsFRbNFR+RrnesGXC1cEAjs82MOlvEAW+6nK41e+KRhpkcMhYOU4ltjCMlttGqIenhmsaduOLd3WB4OyIbXwsoEH6ZjsTvWdDyqws4tDp2ZHH+wFahToSUH14/EhgzzsHHJYNKOCHOWr4hE3TPynE+JjdhJbUUsB9NfvrhG7IeDOkGbVzcE6KSxOwwWcwURx2tg0RzdYlQPDFWRjxodoEBoQ5AthDs/9MViMKw91Vy3hZPbsmzBZ0/p2Sv4m8z8pzM2uJI5FI7ROCWvjvvC3jguvicPCevrq6K88pyPTRbFgxppH5rW1du80EPuHdQBpE1Ety4qY2y4Ls84Lk8EafZk/J59WDkZgNnxoVNjMFfKaDqzFrJM/Lqs6urPGqfmVe35NITQ6BrN8+Xbhl2ksfS2hgCqCGZRtwkOBpNz0CnfTto0jGqB6jzdPXkbnqfAKfaVLYsvVfswOWgbfX2AlbykaEjIfavkbDOW9kQLWivdzKUic/rnVFwlFZ5RRokiwrUy1E7TXtY7VI6JTRpWzS0g+AtnyTdo+csB/FVTmq3z8wB8EqQuEXDWA8PriYa3eWHjc9sxFvPlOZjdT2CpgeodLWW1ojXUWssmswhjjHhbbjBgeEVCw+QuYANa2TOohaL2w5WqK/xdu8AxYYTnMUF7VDrYCrN34MQoYc4fhDtxjnwX3379qv/uvnxux9+rv765m9vv5778bid5UDSwo4eWEs2iwWEo/BRrvgsvKKyJilyoeuGOSWYLQPlNZl+Zh8gF4RP1VgH/mhmcnQJTzUt/ILMGDMlLp/jZvA5ErcV3x+YuFay2lwnt+S1X7zzI+brt/u3HkPqfY0lSFPnYx6RixE0LUhDegfA+WOi4vcx3awMboGJQrZwmkO0mUP00l3acDwYH6v0CDmfHVkiUxeEhGswaKJFa5waauEgVklAMsSkS3I9NblXy0QKybTJ3HP2+9ub77/7+s3P3/34w+9jvEVRcoKlRR406hiVve+gXG8Kh3y5qISmWIIbIZER8Lp0ivu0SmetGN1PvkplaxXGuyo+LQmaYGox2kxAtLqs6JmC6Atg/ZyGndLgZJOjKCVfopGHq7EXkuIdfXl2Q94guCIn9vWIuTuquqH3EYDk1nrWYDu9ysn16a5O7TqEvcW6v6ktslzSCtQC1sfs2tko66BYZHHkAou6gnELIpSNWBiYk3E5yvN+HS6nxL8fyWQiSVIuIJwWVT+VUATUwzUEBheoPqzsMObw+Hdi1OG/Cza1r8pA4lidkM4Xp8T0/IxNN63sXa5QjX+jdfZYtV78gzK9Sg8dkOXohQX88np/zq6Pf1iVJxqGF+4cEpapzdboLlwwWLhUAIIPofps/Pz3xHK/P+geziLcM8W74SrZhxbvzn/nLvxMSnZPkPyESz8AIjm/J6SJ5/6+F9b3NszQehcbuvHPW//wD/Z3hY6OGrJsXh+8MHgQ+FCNsdzTrmNZ8R9ZURz/HILKsyXFFsikrrgfi4SXBDJMZgcvH1pwnVj/0N5jVFl770wfJ1Kn0eS5PTJb49lBr8nL2VWsp9HaT6xnWKvfoWU2XouRB6Y2rbw7Q361FIaL4VS6+kvrBUTTdFG3vMeTxbsUVeoKkCHKzmAFFd+A5LN4PyP+NrS2xT2gYGFkOtXLeCPtBdxIAxc7LUABF38iz0jxKbzDq/XapU7xoefkGQI7gTNxagFo/LrAVFh7DpkMIK1RuCOVuxTH8qlg2yppIYJa+jdAAhTk4Ndnbt8Xv5EOH6PB36aKZhhbItUF7VlOpMxZoFEa/6lj5pkXO3KhtBu05BkzLNoQ3GvEOCpcEDiJrl4vn2Wv5BYqdXza6Xfa+Q+EBx61tRAzY2JrqT4CNUcZ1MdUAeSTgpAnTjeGHC2xlWfQn39w1cR5yTleMvUUkOTk2TN/FA+YIsFR9QPR3G2TrJA9E2lCE6wSWnAN/G/jKoUi/y7M60rksg8ktQWMhWgluF/Xn1+dEe8h9rVsJQQgTzAQF5b46eItX/8bgz2/YWnjbN4zfV2Sl2gALDjp4II6juqp4XgReua0PuyWjtNFgul1GTmJi75gvL4nSruSXF18AKbOY8lFFd7e/PjVt9VXP/7l5vu3P799emRhWbosBJIW6nr+pYc0PZxH0ljn0gI/SH8Jc351eiwls3esQ4kcCOpmqEO1zpmDOT0Qf2OyXEDGUjAFODr9vXgbp693UjMxzbXbUN+Hptkn4UT4Hy0Qcsgrn8S1lvPkfpRLk2eTZKttjNIXrpbxd9+QCfE9LPYRtE0wTOUvs0IAJunpoCfHa6f3RpANfSaePs4DyAlraa9hbpf18gm985fGzmnLeUzVFeOdVlHCBkOG4bTXQ9H4B7MafgvnawbJXKnaxkAXy9YLVpa6wHXpR0TlpkBPY2A7PpO4ZtChiLZtFLi0cQZ7MXpKXvP/kMkntkKnbV3UsgXKiRl4NDRY15tjhf8vUZpd/D9QSwECFAAUAAAACACpDFldikUhez4AAAA8AAAAFAAAAAAAAAAAAAAAgAEAAAAAYjFfc3R1ZHkvX19pbml0X18ucHlQSwECFAAUAAAACACpDFldKwJLRJ8MAACxJAAAEAAAAAAAAAAAAAAAgAFwAAAAYjFfc3R1ZHkvZGF0YS5weVBLAQIUABQAAAAIAKkMWV07Hjn/vQwAAAopAAARAAAAAAAAAAAAAACAAT0NAABiMV9zdHVkeS9tb2RlbC5weVBLAQIUABQAAAAIAKkMWV3k8npMlAYAACARAAAPAAAAAAAAAAAAAACAASkaAABiMV9zdHVkeS9ydW4ucHlQSwECFAAUAAAACACpDFldxLlbX64WAAAWTAAAEQAAAAAAAAAAAAAAgAHqIAAAYjFfc3R1ZHkvdHJhaW4ucHlQSwUGAAAAAAUABQA7AQAAxzcAAAAA')
assert hashlib.sha256(raw).hexdigest()=='94dca70239fa1a17ee12dcb8103cba3388fbe992fa58ef633d17c2e0f8e51360'
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name,wanted in EXPECTED_STUDY.items():
        data=archive.read(name);assert hashlib.sha256(data).hexdigest()==wanted
        dest=CODE/name;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
sys.path.insert(0,str(CODE))
from phase4_cjpe.common import atomic_json,sha
PERSIST=Path('/kaggle/working/CS781_B1_Hierarchy_Depth');PERSIST.mkdir(parents=True,exist_ok=True)
WORK=Path('/kaggle/temp/cs781_b1_study');WORK.mkdir(parents=True,exist_ok=True)
(PERSIST/'STUDY_SOURCE.zip').write_bytes(raw)
atomic_json(PERSIST/'SOURCE_MANIFEST.json',dict(vendor=EXPECTED_VENDOR,study=EXPECTED_STUDY))
FIXED=CODE/'FIXED_HOLDOUT_IDS.json';atomic_json(FIXED,['1951_70', '1952_23', '1952_50', '1953_36', '1954_175', '1954_39', '1955_23', '1955_59', '1955_88', '1958_82', '1959_124', '1959_145', '1959_21', '1960_31', '1961_113', '1961_187', '1961_23', '1961_46', '1961_98', '1962_203', '1962_310', '1962_55', '1962_89', '1963_111', '1963_151', '1963_164', '1963_259', '1963_280', '1964_117', '1964_119', '1964_123', '1964_191', '1964_324', '1964_61', '1964_81', '1965_127', '1965_176', '1965_322', '1965_47', '1967_12', '1967_159', '1967_171', '1967_226', '1967_309', '1967_72', '1968_230', '1968_248', '1968_4', '1969_136', '1969_311', '1969_339', '1970_176', '1970_320', '1971_286', '1971_522', '1971_536', '1971_551', '1971_586', '1972_202', '1972_247', '1972_412', '1972_557', '1972_60', '1972_81', '1973_151', '1973_267', '1973_80', '1974_420', '1974_77', '1975_152', '1975_28', '1975_413', '1975_499', '1975_64', '1976_197', '1976_502', '1976_72', '1977_113', '1977_132', '1977_184', '1977_51', '1977_77', '1978_110', '1978_271', '1978_277', '1978_386', '1979_227', '1979_422', '1980_275', '1980_379', '1980_38', '1980_458', '1981_19', '1981_28', '1981_319', '1981_343', '1981_37', '1982_116', '1983_114', '1983_124', '1983_289', '1983_302', '1983_318', '1983_344', '1984_10', '1984_102', '1984_141', '1984_153', '1984_186', '1985_370', '1985_42', '1985_61', '1986_100', '1986_144', '1986_18', '1986_206', '1986_241', '1986_341', '1986_6', '1987_176', '1987_180', '1987_206', '1987_221', '1987_551', '1987_577', '1987_74', '1988_109', '1988_16', '1988_230', '1988_333', '1988_382', '1988_418', '1988_70', '1989_140', '1989_167', '1989_46', '1990_12', '1990_134', '1990_215', '1990_254', '1990_380', '1990_481', '1990_590', '1990_630', '1990_642', '1990_66', '1990_69', '1990_711', '1991_138', '1991_16', '1991_167', '1991_201', '1991_279', '1991_311', '1991_376', '1991_462', '1991_71', '1992_225', '1992_495', '1992_652', '1992_67', '1992_715', '1992_723', '1993_139', '1993_200', '1993_290', '1993_432', '1993_494', '1993_544', '1993_608', '1993_653', '1993_696', '1993_717', '1993_87', '1993_93', '1994_226', '1994_295', '1994_501', '1994_692', '1994_909', '1994_980', '1995_1136', '1995_1264', '1995_194', '1995_266', '1995_468', '1995_488', '1995_526', '1995_547', '1995_549', '1995_761', '1995_808', '1995_972', '1996_1052', '1996_1080', '1996_1345', '1996_149', '1996_1529', '1996_156', '1996_1813', '1996_2058', '1996_364', '1996_406', '1996_412', '1996_471', '1996_555', '1996_556', '1996_559', '1996_661', '1996_69', '1996_889', '1996_93', '1997_1176', '1997_1183', '1997_1247', '1997_1310', '1997_1500', '1997_1537', '1997_1588', '1997_1589', '1997_1605', '1997_1626', '1997_216', '1997_275', '1997_358', '1997_363', '1997_418', '1997_446', '1997_501', '1997_566', '1997_61', '1997_617', '1997_730', '1997_795', '1997_803', '1997_880', '1997_92', '1997_924', '1998_1160', '1998_218', '1998_395', '1998_475', '1998_879', '1998_997', '1999_1072', '1999_1089', '1999_1090', '1999_119', '1999_152', '1999_186', '1999_443', '1999_444', '1999_489', '1999_545', '1999_65', '1999_686', '1999_76', '1999_947', '2000_1066', '2000_1155', '2000_1195', '2000_1504', '2000_1521', '2000_208', '2000_259', '2000_752', '2000_811', '2000_853', '2000_877', '2001_1030', '2001_1049', '2001_1112', '2001_1159', '2001_121', '2001_265', '2001_557', '2001_574', '2001_598', '2001_791', '2001_792', '2001_851', '2001_872', '2001_990', '2002_11', '2002_1147', '2002_1298', '2002_1302', '2002_1308', '2002_181', '2002_234', '2002_245', '2002_403', '2002_41', '2002_414', '2002_44', '2002_48', '2002_505', '2002_621', '2002_698', '2002_708', '2002_71', '2002_789', '2002_968', '2003_1073', '2003_1316', '2003_150', '2003_266', '2003_429', '2003_45', '2003_533', '2003_557', '2003_643', '2003_713', '2003_724', '2003_799', '2003_831', '2003_957', '2003_994', '2004_1029', '2004_1107', '2004_1171', '2004_170', '2004_184', '2004_332', '2004_470', '2004_606', '2004_854', '2004_928', '2005_106', '2005_285', '2005_41', '2005_415', '2005_417', '2005_469', '2005_570', '2005_887', '2006_1046', '2006_1050', '2006_1124', '2006_1211', '2006_287', '2006_447', '2006_468', '2006_469', '2006_584', '2006_781', '2006_824', '2006_88', '2006_975', '2006_981', '2007_1086', '2007_1220', '2007_1261', '2007_1356', '2007_137', '2007_1407', '2007_256', '2007_260', '2007_360', '2007_397', '2007_409', '2007_410', '2007_44', '2007_460', '2007_5', '2007_540', '2007_565', '2007_637', '2007_657', '2007_753', '2007_792', '2007_916', '2007_930', '2008_1027', '2008_1430', '2008_1459', '2008_1550', '2008_1941', '2008_195', '2008_2053', '2008_2086', '2008_2261', '2008_2447', '2008_2456', '2008_2469', '2008_2542', '2008_26', '2008_268', '2008_2686', '2008_369', '2008_408', '2008_433', '2008_63', '2009_1018', '2009_1095', '2009_1623', '2009_163', '2009_1684', '2009_1693', '2009_1794', '2009_1858', '2009_2122', '2009_217', '2009_221', '2009_293', '2009_299', '2009_353', '2009_391', '2009_542', '2009_619', '2009_762', '2009_961', '2010_1020', '2010_1172', '2010_1250', '2010_1266', '2010_334', '2010_436', '2010_505', '2010_523', '2010_549', '2010_6', '2010_64', '2010_696', '2010_72', '2010_890', '2011_103', '2011_1058', '2011_130', '2011_254', '2011_397', '2011_474', '2011_476', '2011_551', '2011_623', '2011_689', '2011_802', '2011_822', '2011_93', '2011_931', '2011_99', '2012_142', '2012_785', '2012_795', '2013_1008', '2013_127', '2013_136', '2013_493', '2013_61', '2013_761', '2013_780', '2013_982', '2014_162', '2014_331', '2014_416', '2014_672', '2014_800', '2015_259', '2015_314', '2015_327', '2015_442', '2015_55', '2015_605', '2015_801', '2016_122', '2016_172', '2016_306', '2016_360', '2016_393', '2016_495', '2016_568', '2016_591', '2016_595', '2016_660', '2016_692', '2016_76', '2017_112', '2017_115', '2017_137', '2017_164', '2017_224', '2017_297', '2017_323', '2017_461', '2017_573', '2018_1037', '2018_238', '2018_245', '2018_35', '2018_406', '2018_57', '2018_617', '2018_625', '2018_701', '2019_1089', '2019_114', '2019_255', '2019_408', '2019_419', '2019_459', '2019_484', '2019_485', '2019_527', '2019_55', '2019_675', '2019_785', '2019_786', '2019_805', '2019_946', '2019_999', '2020_182', '2020_313'])
EXPECTED_RECOVERED={'Train/train.json':{'sha256':'49bfc5f4d441833e2468e4c280c390ce361e2bfaffd5fbf1ad54157cddda164c','bytes':611284455},'Train/dev.json':{'sha256':'a071b080d3ffcd9789009cf46f6e4f1008142de0f44db9b97df9b64b6ee201c9','bytes':22387465},'Test/test.json':{'sha256':'88913c10c288a6f77a030d0391904d40b645fc242f2774f0ef8216e9f5afa792','bytes':35050736}}
recovered=[]
for p in Path('/kaggle/input').rglob('RECOVERY_MANIFEST.json'):
    try:
        if json.loads(p.read_text()).get('files')==EXPECTED_RECOVERED:recovered.append(p)
    except (ValueError,OSError):pass
assert len(recovered)==1,'Corrected recovered data unavailable or ambiguous.'
DATA=recovered[0].parent
for relative,wanted in EXPECTED_RECOVERED.items():
    assert (DATA/relative).stat().st_size==wanted['bytes'] and sha(DATA/relative)==wanted['sha256']
bundles=[]
for p in Path('/kaggle/input').rglob('INPUT_MANIFEST.json'):
    try:
        if json.loads(p.read_text()).get('family')=='b1':bundles.append(p.parent)
    except (ValueError,OSError):pass
assert len(bundles)==1,'Attach exact own Phase1 B1 input.'
BUNDLE=bundles[0]
assert sha(BUNDLE/'checkpoint.pt')=='d93a2f1796e90b9fdbe4a7b542f76eb6065dab619feed4a1ef5a9952e2376a1f'
assert sha(BUNDLE/'tokenizer.json')=='5c0bd30290e0bd6919253b1ff864f52053924808aabd33482df92cc4bd471c8e'
assert shutil.disk_usage(PERSIST).free>8*1024**3,'Need8GiB for best/latest states and exports.'
atomic_json(PERSIST/'INPUT_PROVENANCE.json',dict(data=EXPECTED_RECOVERED,bundle=str(BUNDLE),parent_sha256=sha(BUNDLE/'checkpoint.pt')))
CONTROL={'experiment_stage': 'head_ablations', 'segment': 2, 'resume_dataset': 'krshnakarthikeyagn/cs781-b1-hierarchy-resume-20261022', 'expected_resume_manifest': '137141c029015c85b7f84410944f91cc2b570209f518dc09e8f578309e14e3c4', 'session_hours': 4.2}
RESUME_ROOTS={}
if CONTROL.get('resume_dataset'):
    candidates=[]
    for p in Path('/kaggle/input').rglob('RESUME_MANIFEST.json'):
        if sha(p)==CONTROL['expected_resume_manifest']:candidates.append(p)
    assert len(candidates)==1,'Exact verified previous-segment resume manifest required.'
    manifest_path=candidates[0];resume=json.loads(manifest_path.read_text())
    for job in resume['jobs']:
        dest=WORK/'resume'/job['name'];dest.mkdir(parents=True,exist_ok=True)
        for name,record in job['files'].items():
            source=manifest_path.parent/record['filename']
            assert source.stat().st_size==record['bytes'] and sha(source)==record['sha256'],'Resume file mismatch'
            target=dest/name
            if not target.exists():target.symlink_to(source)
        RESUME_ROOTS[job['name']]=str(dest)
print('B1_STUDY_INPUTS_VERIFIED',{'bundle':str(BUNDLE),'data':str(DATA)})


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2,'Two isolated GPUs are required for this comparison.'
print('ALLOCATED_GPUS',[(i,torch.cuda.get_device_name(i)) for i in range(2)])
BASE=dict(seed=781,max_epochs=4,accumulation=8,warmup_updates=100,patience=5,backbone_lr=1e-5,head_lr=3e-4,
          encoder_depth=6,expand_ffn=False,chunk_batch=4,segment=CONTROL['segment'],segment_hours=3.,resume_root=None,
          bundle=str(BUNDLE),data_root=str(DATA),fixed_holdout=str(FIXED),session_end_unix=SESSION_END)
definitions=[dict(name='B1_FullData_Reference',representation='reference',head_mode='reference',max_tokens=256,chunks_per_case=8,overlap=0,window_policy='uniform'),
             dict(name='B1_DocumentHierarchy',representation='windows',head_mode='transformer_gru',max_tokens=512,chunks_per_case=16,overlap=100,window_policy='tail')]
if CONTROL.get('experiment_stage', 'baseline') == 'head_ablations':
    definitions=[dict(name='B1_Window_GRUOnly',representation='windows',head_mode='gru_only',max_tokens=512,chunks_per_case=16,overlap=100,window_policy='tail'),
                 dict(name='B1_Window_TransformerOnly',representation='windows',head_mode='transformer_only',max_tokens=512,chunks_per_case=16,overlap=100,window_policy='tail')]
JOBS=[]
for definition in definitions:
    config=dict(BASE,**definition)
    # Resume manifests contain only unfinished jobs; completed jobs must not restart.
    if CONTROL.get('resume_dataset') and config['name'] not in RESUME_ROOTS:
        continue
    config.update(run_root=str(PERSIST/'runs'/config['name']),work_root=str(WORK/config['name']))
    config['resume_root']=RESUME_ROOTS.get(config['name'])
    JOBS.append(config)
assert JOBS, 'Resume manifest does not match any scheduled unfinished experiment.'
atomic_json(PERSIST/'RUN_SCHEDULE.json',JOBS)
from b1_study.run import launch
RESULTS=launch(JOBS,PERSIST)
print('SEGMENT_FINISHED',[(r['name'],r['state'],r.get('optimizer_updates'),r.get('completed_epochs')) for r in RESULTS])
